# 05 - Mean Filter: Kernel Size, Normalization, and Borders

For a \(k\times k\) mean filter, each coefficient is \(1/k^2\). The interactive control studies kernel size, while the static comparison reports border sensitivity and edge loss.

In [4]:
from pathlib import Path
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_course_root() -> Path:
    """Locate the cloned course repository without a machine-specific path."""
    candidates = []
    configured = os.environ.get("VISION_ROBOTICA_REPO")
    if configured:
        candidates.append(Path(configured).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "imagenes").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Course repository not found. Run the notebook from the cloned repository root "
        "or define VISION_ROBOTICA_REPO."
    )

ROOT = find_course_root()
COURSE_IMAGES = ROOT / "imagenes"
STUDENT_IMAGES = ROOT / "student_work" / "imagenes"
STUDENT_IMAGES.mkdir(parents=True, exist_ok=True)

def load_rgb(filename: str) -> np.ndarray:
    path = COURSE_IMAGES / filename
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Official course image not found: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print(f"Course root: {ROOT}")
print(f"Official images: {COURSE_IMAGES}")
print(f"Student images: {STUDENT_IMAGES}")

Course root: C:\Users\20808\Documents\Repositorios\Vision_en_Robotica
Official images: C:\Users\20808\Documents\Repositorios\Vision_en_Robotica\imagenes
Student images: C:\Users\20808\Documents\Repositorios\Vision_en_Robotica\student_work\imagenes


In [5]:
from ipywidgets import IntSlider, Dropdown, interactive_output, VBox, HBox
from IPython.display import display

image_rgb = load_rgb("lenna.png")
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

border_modes = {
    "Reflect 101": cv2.BORDER_REFLECT_101,
    "Reflect": cv2.BORDER_REFLECT,
    "Replicate": cv2.BORDER_REPLICATE,
    "Constant zero": cv2.BORDER_CONSTANT,
}

def show_mean_filter(kernel_size: int, border_name: str) -> None:
    filtered = cv2.blur(gray, (kernel_size, kernel_size), borderType=border_modes[border_name])
    difference = cv2.absdiff(gray, filtered)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, image, title in zip(
        axes,
        [gray, filtered, difference],
        ["Original", f"Mean {kernel_size}x{kernel_size}", "Absolute change"],
    ):
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

kernel_control = IntSlider(value=5, min=3, max=21, step=2, description="Kernel")
border_control = Dropdown(options=list(border_modes), value="Reflect 101", description="Border")
output = interactive_output(show_mean_filter, {"kernel_size": kernel_control, "border_name": border_control})
display(VBox([HBox([kernel_control, border_control]), output]))

In [6]:
def gradient_energy(image: np.ndarray) -> float:
    gx = cv2.Sobel(image, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(image, cv2.CV_32F, 0, 1)
    return float(np.mean(cv2.magnitude(gx, gy)))

baseline = gradient_energy(gray)
print(f"{'Kernel':>8s} {'Std':>9s} {'Gradient retention':>20s} {'Border difference':>20s}")
for kernel_size in (3, 5, 9, 15, 21):
    reflect = cv2.blur(gray, (kernel_size, kernel_size), borderType=cv2.BORDER_REFLECT_101)
    constant = cv2.blur(gray, (kernel_size, kernel_size), borderType=cv2.BORDER_CONSTANT)
    border = kernel_size // 2
    border_mask = np.zeros_like(gray, dtype=bool)
    border_mask[:border, :] = border_mask[-border:, :] = True
    border_mask[:, :border] = border_mask[:, -border:] = True
    border_difference = np.mean(cv2.absdiff(reflect, constant)[border_mask])
    print(f"{kernel_size:8d} {reflect.std():9.2f} {gradient_energy(reflect)/baseline:20.3f} {border_difference:20.3f}")

  Kernel       Std   Gradient retention    Border difference
       3     46.91                0.773               39.918
       5     46.02                0.626               35.995
       9     44.43                0.478               33.431
      15     42.29                0.370               32.225
      21     40.38                0.309               31.793


## Interpretation

Kernel size controls the spatial scale of averaging. Border mode is most visible near the image boundary and becomes more influential as the kernel grows. Record a kernel that suppresses unwanted local variation while retaining the smallest feature required by the robot task.